In [1]:
import pandas as pd
import numpy as np
import re
import csv
import os

## generate 3-to-1 letter AA codes

In [2]:
# check if the genes of interest are in catalog df
protein_sequences_file = './data/catalog/protein_sequences.csv'
protein_sequences_df = pd.read_csv(protein_sequences_file)
# Extract genes of interest
genes_of_interest = protein_sequences_df['gene'].unique()

In [3]:
## load the 2021 complete excel file
# Load the WHO catalog
who_catalog_path = './data/catalog/WHO-UCN-GTB-PCI-2021.7-eng.xlsx'  # Update the path as necessary
who_catalog = pd.read_excel(who_catalog_path, sheet_name='Mutation_catalogue',header=1)

In [4]:
who_catalog = pd.read_excel(who_catalog_path, sheet_name='Mutation_catalogue',header=1)
who_catalog.columns

Index(['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3',
       'algorithm pass', 'Present_SOLO_R', 'Present_SOLO_SR', 'Present_S',
       'Absent_S', 'Present_R', 'Absent_R', 'PPV', 'PPV_lb', 'PPV_ub',
       'PPV | SOLO', 'PPV | SOLO_lb', 'PPV | SOLO_ub', 'Sensitivity',
       'Sensitivity_lb', 'Sensitivity_ub', 'Specificity', 'Specificity_lb',
       'Specificity_ub', 'LR+', 'LR+_lb', 'LR+_ub', 'LR-', 'LR-_lb', 'LR-_ub',
       'OR', 'OR_lb', 'OR_ub', 'OR SOLO', 'OR SOLO_lb', 'OR SOLO_ub',
       'OR SOLO_FE-sig', 'Neutral masked', 'Unnamed: 37', 'Unnamed: 38',
       'Miotto et al. (PMID 29284687)', 'NGS Guide 2018',
       'Level of resistance to INH or MXF', 'RIF CC guide 2021',
       'Hain GenoType MTBDRplus V2.0', 'Nipro Genoscholar NTM+MDRTB II',
       'Cepheid Xpert MTB/RIF', 'Cepheid Xpert MTB/RIF Ultra',
       'Hain GenoType MTBDRsl V2.0', 'Cepheid Xpert MTB/XDR',
       'Nipro Genoscholar PZA-TB II', 'Unnamed: 50', 'Unnamed: 51'],
      dtype='object')

In [5]:
who_catalog = pd.read_excel(who_catalog_path, sheet_name='Mutation_catalogue',header=0)
who_catalog.columns

Index(['drug', 'tier', 'variant (common_name)', 'Genome position', 'DATASET',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9',
       'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13',
       'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17',
       'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21',
       'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25',
       'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29',
       'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33',
       'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36',
       'INITIAL CONFIDENCE GRADING', 'DATASET(S)', 'Previous WHO guidance',
       'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'WHO-endorsed gDST assays',
       'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47',
       'Unnamed: 48', 'Unnamed: 49', 'Additional grading criteria',
       'FINAL CONFIDENCE GRADING'],
      dtype='object')

In [6]:
# Flatten the column names if they are multi-level
if isinstance(who_catalog.columns, pd.MultiIndex):
    who_catalog.columns = ['_'.join(filter(None, col)).strip() for col in who_catalog.columns.values]


In [7]:
# Step 1: Read the file and flatten multi-index columns if present
who_catalog = pd.read_excel(who_catalog_path, sheet_name='Mutation_catalogue', header=[0, 1])  # Use header=[0, 1] to capture multi-index columns

# Step 2: Flatten the multi-level columns
who_catalog.columns = ['_'.join(filter(None, col)).strip() for col in who_catalog.columns]

# Step 3: Check the flattened columns
print(who_catalog.columns)


Index(['drug_Unnamed: 0_level_1', 'tier_Unnamed: 1_level_1',
       'variant (common_name)_Unnamed: 2_level_1',
       'Genome position_Unnamed: 3_level_1', 'DATASET_algorithm pass',
       'DATASET_Present_SOLO_R', 'DATASET_Present_SOLO_SR',
       'DATASET_Present_S', 'DATASET_Absent_S', 'DATASET_Present_R',
       'DATASET_Absent_R', 'DATASET_PPV', 'DATASET_PPV_lb', 'DATASET_PPV_ub',
       'DATASET_PPV | SOLO', 'DATASET_PPV | SOLO_lb', 'DATASET_PPV | SOLO_ub',
       'DATASET_Sensitivity', 'DATASET_Sensitivity_lb',
       'DATASET_Sensitivity_ub', 'DATASET_Specificity',
       'DATASET_Specificity_lb', 'DATASET_Specificity_ub', 'DATASET_LR+',
       'DATASET_LR+_lb', 'DATASET_LR+_ub', 'DATASET_LR-', 'DATASET_LR-_lb',
       'DATASET_LR-_ub', 'DATASET_OR', 'DATASET_OR_lb', 'DATASET_OR_ub',
       'DATASET_OR SOLO', 'DATASET_OR SOLO_lb', 'DATASET_OR SOLO_ub',
       'DATASET_OR SOLO_FE-sig', 'DATASET_Neutral masked',
       'INITIAL CONFIDENCE GRADING_Unnamed: 37_level_1',
       'DA

In [8]:
# Step 1: Select the columns with the correct names
catalog_df = who_catalog[[
    'drug_Unnamed: 0_level_1', 
    'variant (common_name)_Unnamed: 2_level_1', 
    'DATASET_Present_R', 
    'DATASET_Present_S', 
    'FINAL CONFIDENCE GRADING_Unnamed: 51_level_1'
]]

# Step 2: Rename the columns to more meaningful names (optional)
catalog_df.columns = ['drug', 'variant', 'Present_R', 'Present_S', 'FINAL_CONFIDENCE_GRADING']

# Step 3: Check the first few rows of the selected data
print(catalog_df.head())


  drug     variant  Present_R  Present_S   FINAL_CONFIDENCE_GRADING
0  AMI  rrs_a1401g      939.0       50.0               1) Assoc w R
1  AMI   eis_c-14t       32.0       51.0               1) Assoc w R
2  AMI  rrs_g1484t        6.0        2.0     2) Assoc w R - Interim
3  AMI  rrs_c1402t        5.0       10.0     2) Assoc w R - Interim
4  AMI  whiB6_A77V        3.0       97.0  3) Uncertain significance


In [9]:
# Extract relevant columns

# Filter out rows with NaN values in the 'drug' or 'variant' columns
catalog_df = catalog_df.dropna(subset=['drug', 'variant'])

# subset for target genes
filtered_df = catalog_df[catalog_df['variant'].str.startswith(tuple(genes_of_interest))]

# Display the filtered DataFrame
print(len(filtered_df))
# Discard rows that have 'ins' or 'del' in the variant column
filtered_df = filtered_df[~filtered_df['variant'].str.contains('ins|del')]

6576


In [10]:
filtered_df

,drug,variant,Present_R,Present_S,FINAL_CONFIDENCE_GRADING
1576,BDQ,Rv0678_A36V,2.0,0.0,3) Uncertain significance
1581,BDQ,Rv0678_A99P,1.0,0.0,3) Uncertain significance
1582,BDQ,Rv0678_F19S,1.0,0.0,3) Uncertain significance
1583,BDQ,Rv0678_G121R,1.0,0.0,3) Uncertain significance
1584,BDQ,Rv0678_I67L,1.0,0.0,3) Uncertain significance
...,...,...,...,...,...
17383,STM,gid_E92D,1390.0,1041.0,5) Not assoc w R
17385,STM,gid_L16R,222.0,949.0,5) Not assoc w R
17390,STM,gid_Y195H,34.0,139.0,5) Not assoc w R
17393,STM,rpsL_c-259t (Rv0681_194),0.0,50.0,5) Not assoc w R


In [11]:
filtered_df=filtered_df.rename(columns={"FINAL CONFIDENCE GRADING": "confidence"})

In [12]:

# Step 1: Extract the gene names
filtered_df['gene'] = filtered_df['variant'].str.split('_').str[0]

In [13]:
filtered_df

,drug,variant,Present_R,Present_S,FINAL_CONFIDENCE_GRADING,gene
1576,BDQ,Rv0678_A36V,2.0,0.0,3) Uncertain significance,Rv0678
1581,BDQ,Rv0678_A99P,1.0,0.0,3) Uncertain significance,Rv0678
1582,BDQ,Rv0678_F19S,1.0,0.0,3) Uncertain significance,Rv0678
1583,BDQ,Rv0678_G121R,1.0,0.0,3) Uncertain significance,Rv0678
1584,BDQ,Rv0678_I67L,1.0,0.0,3) Uncertain significance,Rv0678
...,...,...,...,...,...,...
17383,STM,gid_E92D,1390.0,1041.0,5) Not assoc w R,gid
17385,STM,gid_L16R,222.0,949.0,5) Not assoc w R,gid
17390,STM,gid_Y195H,34.0,139.0,5) Not assoc w R,gid
17393,STM,rpsL_c-259t (Rv0681_194),0.0,50.0,5) Not assoc w R,rpsL


In [14]:
filtered_df['one_letter_mutation'] = filtered_df['variant'].str.extract(r'_(\w+\d+\w+)')


In [15]:
# Step 2: Filter rows that match the valid mutation pattern (e.g., "A123B")
valid_pattern = r'^\w+_[A-Z]\d+[A-Z]$'
filtered_df = filtered_df[filtered_df['variant'].str.match(valid_pattern)]


In [16]:
filtered_df

,drug,variant,Present_R,Present_S,FINAL_CONFIDENCE_GRADING,gene,one_letter_mutation
1576,BDQ,Rv0678_A36V,2.0,0.0,3) Uncertain significance,Rv0678,A36V
1581,BDQ,Rv0678_A99P,1.0,0.0,3) Uncertain significance,Rv0678,A99P
1582,BDQ,Rv0678_F19S,1.0,0.0,3) Uncertain significance,Rv0678,F19S
1583,BDQ,Rv0678_G121R,1.0,0.0,3) Uncertain significance,Rv0678,G121R
1584,BDQ,Rv0678_I67L,1.0,0.0,3) Uncertain significance,Rv0678,I67L
...,...,...,...,...,...,...,...
17318,STM,gid_A27P,0.0,5.0,3) Uncertain significance,gid,A27P
17379,STM,gid_V110G,NaN,NaN,4) Not assoc w R - Interim,gid,V110G
17383,STM,gid_E92D,1390.0,1041.0,5) Not assoc w R,gid,E92D
17385,STM,gid_L16R,222.0,949.0,5) Not assoc w R,gid,L16R


In [17]:
## drop invalid mutations
filtered_df = filtered_df.dropna(subset=['one_letter_mutation'])

In [18]:
print(np.unique(filtered_df['gene']))

['Rv0678' 'atpE' 'ddn' 'embB' 'ethA' 'fgd1' 'gid' 'gyrA' 'gyrB' 'inhA'
 'katG' 'pepQ' 'pncA' 'rplC' 'rpoB' 'rpsL' 'tlyA']


In [19]:
filtered_df =filtered_df.drop_duplicates()

In [20]:
filtered_df.to_csv('./data/catalog/2021_mutations_with_one_letter_all_confidence.csv', index=False)

## feature 1: frequency

In [21]:
# Calculate the frequency of each variant in the population
filtered_df['frequency'] = (filtered_df['Present_R']) / (filtered_df['Present_R'] + filtered_df['Present_S'])

In [22]:
filtered_df

,drug,variant,Present_R,Present_S,FINAL_CONFIDENCE_GRADING,gene,one_letter_mutation,frequency
1576,BDQ,Rv0678_A36V,2.0,0.0,3) Uncertain significance,Rv0678,A36V,1.000000
1581,BDQ,Rv0678_A99P,1.0,0.0,3) Uncertain significance,Rv0678,A99P,1.000000
1582,BDQ,Rv0678_F19S,1.0,0.0,3) Uncertain significance,Rv0678,F19S,1.000000
1583,BDQ,Rv0678_G121R,1.0,0.0,3) Uncertain significance,Rv0678,G121R,1.000000
1584,BDQ,Rv0678_I67L,1.0,0.0,3) Uncertain significance,Rv0678,I67L,1.000000
...,...,...,...,...,...,...,...,...
17318,STM,gid_A27P,0.0,5.0,3) Uncertain significance,gid,A27P,0.000000
17379,STM,gid_V110G,NaN,NaN,4) Not assoc w R - Interim,gid,V110G,NaN
17383,STM,gid_E92D,1390.0,1041.0,5) Not assoc w R,gid,E92D,0.571781
17385,STM,gid_L16R,222.0,949.0,5) Not assoc w R,gid,L16R,0.189582


In [23]:
filtered_df.drop(columns=['Present_R','Present_S'], inplace=True)

In [24]:
filtered_df

,drug,variant,FINAL_CONFIDENCE_GRADING,gene,one_letter_mutation,frequency
1576,BDQ,Rv0678_A36V,3) Uncertain significance,Rv0678,A36V,1.000000
1581,BDQ,Rv0678_A99P,3) Uncertain significance,Rv0678,A99P,1.000000
1582,BDQ,Rv0678_F19S,3) Uncertain significance,Rv0678,F19S,1.000000
1583,BDQ,Rv0678_G121R,3) Uncertain significance,Rv0678,G121R,1.000000
1584,BDQ,Rv0678_I67L,3) Uncertain significance,Rv0678,I67L,1.000000
...,...,...,...,...,...,...
17318,STM,gid_A27P,3) Uncertain significance,gid,A27P,0.000000
17379,STM,gid_V110G,4) Not assoc w R - Interim,gid,V110G,NaN
17383,STM,gid_E92D,5) Not assoc w R,gid,E92D,0.571781
17385,STM,gid_L16R,5) Not assoc w R,gid,L16R,0.189582


In [25]:
filtered_df.to_csv('./data/derived_features/2021_all_proteins_frequency_catalog.csv', index=False)

## generate mutated sequences from the WHO catalog

In [41]:
protein_sequences_df = pd.read_csv('./data/catalog/protein_sequences.csv')
# Output directory
output_dir = 'mutated_sequences_2021'
# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

In [42]:
catalog_df=pd.read_csv('./data/derived_features/2021_all_proteins_frequency_catalog.csv')

In [44]:
catalog_df

,drug,variant,FINAL_CONFIDENCE_GRADING,gene,one_letter_mutation,frequency
0,BDQ,Rv0678_A36V,3) Uncertain significance,Rv0678,A36V,1.000000
1,BDQ,Rv0678_A99P,3) Uncertain significance,Rv0678,A99P,1.000000
2,BDQ,Rv0678_F19S,3) Uncertain significance,Rv0678,F19S,1.000000
3,BDQ,Rv0678_G121R,3) Uncertain significance,Rv0678,G121R,1.000000
4,BDQ,Rv0678_I67L,3) Uncertain significance,Rv0678,I67L,1.000000
...,...,...,...,...,...,...
4685,STM,gid_A27P,3) Uncertain significance,gid,A27P,0.000000
4686,STM,gid_V110G,4) Not assoc w R - Interim,gid,V110G,NaN
4687,STM,gid_E92D,5) Not assoc w R,gid,E92D,0.571781
4688,STM,gid_L16R,5) Not assoc w R,gid,L16R,0.189582


In [45]:
# Function to apply a mutation
# produce mutated seq
def apply_mutation(wildtype_seq, mutation):
    # Check if the mutation format is valid (e.g., A36V)
    if len(mutation) < 3 or not mutation[1:-1].isdigit():
        return None
    
    # Extract the original amino acid, position, and new amino acid
    original_aa = mutation[0]  # First character
    position = int(mutation[1:-1])  # Middle part (digits)
    new_aa = mutation[-1]  # Last character

    # Check if the position is valid within the sequence
    if position <= 0 or position > len(wildtype_seq):
        return None

    # Apply the mutation and create the mutated sequence
    mutated_seq = wildtype_seq[:position - 1] + new_aa + wildtype_seq[position:]
    
    # Return the original amino acid, new amino acid, position, and mutated sequence
    return original_aa, new_aa, position, mutated_seq

In [46]:
# Example usage:
wildtype_seq = "MGELVQVLVGIVTGLMGAGK"
mutation = "A36V"
result = apply_mutation(wildtype_seq, mutation)
print(result)  # Output: ('A', 'V', 36, 'MGELVQVLVGIVTGLMGVGK')

None


In [ ]:
# Process each protein sequence and produce mutated seq
for index, row in protein_sequences_df.iterrows():
    gene = row['gene']
    rv_id = row['RV']
    wildtype_seq = row['protein_sequence']
    
    # Output FASTA file for the gene
    output_file = os.path.join(output_dir, f"{rv_id}_{gene}.fasta")
    
    with open(output_file, mode='w') as file:
        # Write wildtype sequence
        file.write(f">{rv_id}|{gene}|Wildtype\n")
        file.write(f"{wildtype_seq}\n")
        
        # Get mutations for the specific gene
        gene_mutations = catalog_df[catalog_df['gene'] == gene]
        
        for _, mut_row in gene_mutations.iterrows():
            mutation = mut_row['one_letter_mutation']
            result = apply_mutation(wildtype_seq, mutation)
            if result:
                original_aa, new_aa, position, mutated_seq = result
                mutation_label = f"{gene}_p.{original_aa}{position}{new_aa}"
                file.write(f">{rv_id}|{gene}|{mutation_label}\n")
                file.write(f"{mutated_seq}\n")
                print(f"Mutation {mutation} applied for gene {gene}.")

## feature 2: compute delta-z value from the mutated seqs

In [55]:
# call delta_z_calculation file

In [48]:
import torch
import numpy as np
import pandas as pd
import esm
from Bio import SeqIO

In [49]:
# Function to calculate embeddings
def get_embedding(sequence):
    sequence = sequence.upper()  # Ensure sequence is in uppercase
    sequence = sequence.replace('*', '')  # Remove any stop codons
    batch_labels, batch_strs, batch_tokens = batch_converter([("sequence", sequence)])
    with torch.no_grad():
        results = model(batch_tokens.to('cuda'), repr_layers=[6])
    token_embeddings = results["representations"][6].cpu().numpy()
    return token_embeddings.mean(axis=1)

In [50]:
# Load the pretrained ESM-2 model
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval().to('cuda')

ESM2(
  (embed_tokens): Embedding(33, 320, padding_idx=1)
  (layers): ModuleList(
    (0-5): 6 x TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=320, out_features=320, bias=True)
        (v_proj): Linear(in_features=320, out_features=320, bias=True)
        (q_proj): Linear(in_features=320, out_features=320, bias=True)
        (out_proj): Linear(in_features=320, out_features=320, bias=True)
        (rot_emb): RotaryEmbedding()
      )
      (self_attn_layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=320, out_features=1280, bias=True)
      (fc2): Linear(in_features=1280, out_features=320, bias=True)
      (final_layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
    )
  )
  (contact_head): ContactPredictionHead(
    (regression): Linear(in_features=120, out_features=1, bias=True)
    (activation): Sigmoid()
  )
  (emb_layer_norm_after): LayerNorm((320,), eps=1e-05, elementwis

In [51]:
# Directory containing the FASTA files
fasta_dir = '/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/mutated_sequences_2021'

# Output CSV file
output_file = '/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021_all_delta_z_values.csv'

In [52]:
# Initialize the results list
results = []

# Process each FASTA file
for fasta_file in os.listdir(fasta_dir):
    if fasta_file.endswith('.fasta'):
        file_path = os.path.join(fasta_dir, fasta_file)
        records = list(SeqIO.parse(file_path, "fasta"))
        print("loaded file: ", file_path)
        
        # Extract RV and gene names from the file name
        rv_id, gene_name = fasta_file.replace('.fasta', '').split('_')
        print("continue")
        
        # Find the wildtype sequence
        wildtype_record = None
        for record in records:
            if "Wildtype" in record.description:
                wildtype_record = record
                break
        
        if wildtype_record is None:
            print(f"No wildtype sequence found in {fasta_file}")
            continue
        
        wildtype_seq = str(wildtype_record.seq).upper()
        wildtype_embedding = get_embedding(wildtype_seq)
        
        for record in records:
            if record.description == wildtype_record.description:
                continue
            mutation = record.description
            mutated_seq = str(record.seq).upper().replace('*', '')
            try:
                mutated_embedding = get_embedding(mutated_seq)
                delta_z = np.linalg.norm(mutated_embedding - wildtype_embedding)
                results.append([fasta_file, mutation, delta_z, rv_id, gene_name])
            except KeyError as e:
                print(f"Error processing {mutation} in {fasta_file}: {e}")

loaded file:  /work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/mutated_sequences_2021/Rv0005_gyrB.fasta
continue
loaded file:  /work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/mutated_sequences_2021/Rv0006_gyrA.fasta
continue
loaded file:  /work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/mutated_sequences_2021/Rv0407_fgd1.fasta
continue
loaded file:  /work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/mutated_sequences_2021/Rv0667_rpoB.fasta
continue
loaded file:  /work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/mutated_sequences_2021/Rv0678_Rv0678.fasta
continue
loaded file:  /work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/mutated_sequences_2021/Rv0682_rpsL.fasta
continue
loaded file:  /work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/mutated_sequences_2021/Rv0701_rplC.fasta
continue
loaded file:  /work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/mutated_sequences_2021/Rv1305_atpE.fasta
continue
loaded file:  /work/pi_annagre

In [53]:
# Save the results to a CSV file
results_df = pd.DataFrame(results, columns=['filename', 'mutation', 'delta_z', 'rv', 'gene'])
results_df.to_csv(output_file, index=False)

print(f"Delta Z values have been saved to {output_file}.")


Delta Z values have been saved to /work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021_all_delta_z_values.csv.


In [54]:
results_df

,filename,mutation,delta_z,rv,gene
0,Rv0005_gyrB.fasta,Rv0005|gyrB|gyrB_p.E501D,0.033919,Rv0005,gyrB
1,Rv0005_gyrB.fasta,Rv0005|gyrB|gyrB_p.D461N,0.053480,Rv0005,gyrB
2,Rv0005_gyrB.fasta,Rv0005|gyrB|gyrB_p.A504V,0.147652,Rv0005,gyrB
3,Rv0005_gyrB.fasta,Rv0005|gyrB|gyrB_p.N499D,0.028086,Rv0005,gyrB
4,Rv0005_gyrB.fasta,Rv0005|gyrB|gyrB_p.E501V,0.058077,Rv0005,gyrB
...,...,...,...,...,...
4685,Rv3919c_gid.fasta,Rv3919c|gid|gid_p.A27P,0.056275,Rv3919c,gid
4686,Rv3919c_gid.fasta,Rv3919c|gid|gid_p.V110G,0.074505,Rv3919c,gid
4687,Rv3919c_gid.fasta,Rv3919c|gid|gid_p.E92D,0.060223,Rv3919c,gid
4688,Rv3919c_gid.fasta,Rv3919c|gid|gid_p.L16R,0.064680,Rv3919c,gid


In [55]:
## delta-z calculates the euclidean distance between wildtype seq and mutated sequence embeddings
## embeddings are produced from pretrained ESM2(esm2_t6_8M_UR50D)

### add delta-z to the catalog_df

In [77]:
catalog_df=pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021_all_proteins_frequency_catalog.csv')
delta_z_df=pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021_all_delta_z_values.csv')

In [78]:
catalog_df=catalog_df.drop_duplicates()

In [79]:
catalog_df

,drug,variant,FINAL_CONFIDENCE_GRADING,gene,one_letter_mutation,frequency
0,BDQ,Rv0678_A36V,3) Uncertain significance,Rv0678,A36V,1.000000
1,BDQ,Rv0678_A99P,3) Uncertain significance,Rv0678,A99P,1.000000
2,BDQ,Rv0678_F19S,3) Uncertain significance,Rv0678,F19S,1.000000
3,BDQ,Rv0678_G121R,3) Uncertain significance,Rv0678,G121R,1.000000
4,BDQ,Rv0678_I67L,3) Uncertain significance,Rv0678,I67L,1.000000
...,...,...,...,...,...,...
4685,STM,gid_A27P,3) Uncertain significance,gid,A27P,0.000000
4686,STM,gid_V110G,4) Not assoc w R - Interim,gid,V110G,NaN
4687,STM,gid_E92D,5) Not assoc w R,gid,E92D,0.571781
4688,STM,gid_L16R,5) Not assoc w R,gid,L16R,0.189582


In [80]:
delta_z_df

,filename,mutation,delta_z,rv,gene
0,Rv0005_gyrB.fasta,Rv0005|gyrB|gyrB_p.E501D,0.033919,Rv0005,gyrB
1,Rv0005_gyrB.fasta,Rv0005|gyrB|gyrB_p.D461N,0.053480,Rv0005,gyrB
2,Rv0005_gyrB.fasta,Rv0005|gyrB|gyrB_p.A504V,0.147652,Rv0005,gyrB
3,Rv0005_gyrB.fasta,Rv0005|gyrB|gyrB_p.N499D,0.028086,Rv0005,gyrB
4,Rv0005_gyrB.fasta,Rv0005|gyrB|gyrB_p.E501V,0.058077,Rv0005,gyrB
...,...,...,...,...,...
4685,Rv3919c_gid.fasta,Rv3919c|gid|gid_p.A27P,0.056275,Rv3919c,gid
4686,Rv3919c_gid.fasta,Rv3919c|gid|gid_p.V110G,0.074505,Rv3919c,gid
4687,Rv3919c_gid.fasta,Rv3919c|gid|gid_p.E92D,0.060223,Rv3919c,gid
4688,Rv3919c_gid.fasta,Rv3919c|gid|gid_p.L16R,0.064680,Rv3919c,gid


In [81]:
# Ensure the 'Mutation' column is a string and handle missing values
# Remove the gene name from the beginning of the 'Mutation' column
delta_z_df['mutation'] = delta_z_df['mutation'].apply(lambda x: x.split('_')[1] if '_' in x else x)
delta_z_df['mutation'] = delta_z_df['mutation'].astype(str).fillna('')

In [85]:
# Step 1: Remove the 'p.' prefix from the mutation column in delta_z_df
delta_z_df['mutation'] = delta_z_df['mutation'].str.replace('p.', '')
delta_z_df

,filename,mutation,delta_z,rv,gene
0,Rv0005_gyrB.fasta,E501D,0.033919,Rv0005,gyrB
1,Rv0005_gyrB.fasta,D461N,0.053480,Rv0005,gyrB
2,Rv0005_gyrB.fasta,A504V,0.147652,Rv0005,gyrB
3,Rv0005_gyrB.fasta,N499D,0.028086,Rv0005,gyrB
4,Rv0005_gyrB.fasta,E501V,0.058077,Rv0005,gyrB
...,...,...,...,...,...
4685,Rv3919c_gid.fasta,A27P,0.056275,Rv3919c,gid
4686,Rv3919c_gid.fasta,V110G,0.074505,Rv3919c,gid
4687,Rv3919c_gid.fasta,E92D,0.060223,Rv3919c,gid
4688,Rv3919c_gid.fasta,L16R,0.064680,Rv3919c,gid


In [86]:
# Merge the filtered Delta-Z DataFrame with the filtered mutations DataFrame to get the additional columns
merged_df = pd.merge(delta_z_df, catalog_df, left_on=['gene', 'mutation'], right_on=['gene', 'one_letter_mutation'], how='left')

In [87]:
merged_df

,filename,mutation,delta_z,rv,gene,drug,variant,FINAL_CONFIDENCE_GRADING,one_letter_mutation,frequency
0,Rv0005_gyrB.fasta,E501D,0.033919,Rv0005,gyrB,LEV,gyrB_E501D,2) Assoc w R - Interim,E501D,0.538462
1,Rv0005_gyrB.fasta,E501D,0.033919,Rv0005,gyrB,MXF,gyrB_E501D,1) Assoc w R,E501D,0.804878
2,Rv0005_gyrB.fasta,D461N,0.053480,Rv0005,gyrB,LEV,gyrB_D461N,2) Assoc w R - Interim,D461N,0.585366
3,Rv0005_gyrB.fasta,D461N,0.053480,Rv0005,gyrB,MXF,gyrB_D461N,2) Assoc w R - Interim,D461N,0.300000
4,Rv0005_gyrB.fasta,A504V,0.147652,Rv0005,gyrB,LEV,gyrB_A504V,2) Assoc w R - Interim,A504V,0.421053
...,...,...,...,...,...,...,...,...,...,...
5909,Rv3919c_gid.fasta,A27P,0.056275,Rv3919c,gid,STM,gid_A27P,3) Uncertain significance,A27P,0.000000
5910,Rv3919c_gid.fasta,V110G,0.074505,Rv3919c,gid,STM,gid_V110G,4) Not assoc w R - Interim,V110G,NaN
5911,Rv3919c_gid.fasta,E92D,0.060223,Rv3919c,gid,STM,gid_E92D,5) Not assoc w R,E92D,0.571781
5912,Rv3919c_gid.fasta,L16R,0.064680,Rv3919c,gid,STM,gid_L16R,5) Not assoc w R,L16R,0.189582


In [88]:
merged_df=merged_df.drop(columns=['variant','rv','filename'])

In [89]:
merged_df=merged_df.drop_duplicates()

In [90]:
merged_df=merged_df.rename(columns={"FINAL_CONFIDENCE_GRADING": "confidence"})

In [91]:
# Define the desired order of columns, placing 'gene' first
cols = ['gene', 'one_letter_mutation', 'drug', 'confidence','delta_z', 'frequency']

# Reorder the DataFrame columns
reordered_df = merged_df[cols]

# Display or use the reordered DataFrame as needed
reordered_df

,gene,one_letter_mutation,drug,confidence,delta_z,frequency
0,gyrB,E501D,LEV,2) Assoc w R - Interim,0.033919,0.538462
1,gyrB,E501D,MXF,1) Assoc w R,0.033919,0.804878
2,gyrB,D461N,LEV,2) Assoc w R - Interim,0.053480,0.585366
3,gyrB,D461N,MXF,2) Assoc w R - Interim,0.053480,0.300000
4,gyrB,A504V,LEV,2) Assoc w R - Interim,0.147652,0.421053
...,...,...,...,...,...,...
5909,gid,A27P,STM,3) Uncertain significance,0.056275,0.000000
5910,gid,V110G,STM,4) Not assoc w R - Interim,0.074505,NaN
5911,gid,E92D,STM,5) Not assoc w R,0.060223,0.571781
5912,gid,L16R,STM,5) Not assoc w R,0.064680,0.189582


In [93]:
reordered_df.to_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021_all_proteins_freq_deltaz.csv',index=False)

## feature 3: proximity to nearest r-conferring mutations

### helper codes

In [94]:
def extract_position_from_mutation(mutation):
    match = re.search(r'\d+', mutation)
    if match:
        return int(match.group(0))
    return None
def adjust_number(index):
    if index < 30:  # Assuming offset starts at position 23 + 7
        return index + 7
    elif 30 <= index <= 1180:  # Adjusts up to the maximum offset point
        return index + 6
    else:
        return index
def non_self_proximity_r_mutants(gene_subset,unique_valid_r_positions,distance_map):
    # Initialize a list to collect positions causing KeyError
    positions_causing_error = []
    # Initialize the list to store the minimum distances and their indices
    proximity_to_nonself_r_conferring = []
    nearest_mutation_index = []

    for index, row in gene_subset.iterrows():
        current_position = row['position']
        adjusted_current_position = adjust_number(current_position)
        distances = []

        for r_pos in unique_valid_r_positions:
            if r_pos != current_position:
                adjusted_r_pos = adjust_number(r_pos)
                try:
                    dist = distance_map.dist(adjusted_current_position, adjusted_r_pos, raise_na=True)
                    if not np.isnan(dist):
                        distances.append((dist, r_pos))
                except KeyError as e:
                    # Append the positions causing the KeyError
                    positions_causing_error.append((current_position, r_pos))
                    continue  # Continue with the next position

        if distances:
            # Find the entry with the minimum distance
            min_distance, min_index = min(distances, key=lambda x: x[0])
        else:
            min_distance, min_index = np.nan, np.nan

        # Append results to the lists
        proximity_to_nonself_r_conferring.append(min_distance)
        nearest_mutation_index.append(min_index)
    # Store the results in the phenotype_data DataFrame
    gene_subset['Proximity_to_R_Conferring'] = proximity_to_nonself_r_conferring
    gene_subset['Nearest_Mutation_Index'] = nearest_mutation_index
    
    return gene_subset


### compute proximity

In [95]:
from evcouplings.compare import DistanceMap

In [96]:
catalog_df=pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021_all_proteins_freq_deltaz.csv')

In [97]:
# Map confidence levels to phenotypes
def map_confidence_to_phenotype(confidence):
    if confidence in ['1) Assoc w R', '2) Assoc w R - Interim']:
        return 'Resistant'
    elif confidence in ['4) Not assoc w R - Interim', '5) Not assoc w R']:
        return 'Susceptible'
    else:
        return 'Unknown'


In [98]:
catalog_df['phenotype'] =catalog_df['confidence'].apply(map_confidence_to_phenotype)

In [99]:
catalog_df['phenotype'].value_counts()

phenotype
Unknown        4315
Resistant       315
Susceptible      60
Name: count, dtype: int64

In [100]:
catalog_df['position'] =catalog_df['one_letter_mutation'].str.extract(r'(\d+)').astype(int)

In [101]:
# Get the unique combinations of gene and drug
unique_genes_drugs = catalog_df[['gene', 'drug']].drop_duplicates()

# Print each gene and drug pair
for index, row in unique_genes_drugs.iterrows():
    print(f"Gene: {row['gene']}, Drug: {row['drug']}")

Gene: gyrB, Drug: LEV
Gene: gyrB, Drug: MXF
Gene: gyrA, Drug: LEV
Gene: gyrA, Drug: MXF
Gene: fgd1, Drug: DLM
Gene: rpoB, Drug: RIF
Gene: Rv0678, Drug: BDQ
Gene: Rv0678, Drug: CFZ
Gene: rpsL, Drug: STM
Gene: rplC, Drug: LZD
Gene: atpE, Drug: BDQ
Gene: inhA, Drug: ETH
Gene: inhA, Drug: INH
Gene: tlyA, Drug: CAP
Gene: katG, Drug: INH
Gene: pncA, Drug: PZA
Gene: pepQ, Drug: BDQ
Gene: pepQ, Drug: CFZ
Gene: ddn, Drug: DLM
Gene: embB, Drug: EMB
Gene: ethA, Drug: ETH
Gene: gid, Drug: STM


In [102]:
gene_names=list(np.unique(catalog_df['gene']))

In [103]:
##load the protein details file
protein_details_path = '/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/catalog/17_proteins_details.xlsx'  # Update the path as necessary
protein_details = pd.read_excel(protein_details_path, sheet_name='Sheet1')

In [104]:
protein_details

,Drug,gene_name,filename,Uniprot,references,Entry,Protein names,rv,drug_full,Aliases,start_position_on_the_genomic_accession,end_position_on_the_genomic_accession,orientation
0,RIF,rpoB,rpoBC.fasta,RPOB_MYCTU,"WHO 2021 catalog, category 1",P9WGY9,DNA-directed RNA polymerase subunit beta (RNAP...,Rv0667,rifampicin,Rv0667,759806.0,763324.0,plus
1,INH,inhA,FabG1-inhA.fasta,INHA_MYCTU,"WHO 2021 catalog, category 1",P9WGR1,Enoyl-[acyl-carrier-protein] reductase [NADH] ...,Rv1484,isoniazid,Rv1484,1674201.0,1675010.0,plus
2,INH,katG,KatG.fasta,KATG_MYCTU,"WHO 2021 catalog, category 1",P9WIE5,Catalase-peroxidase (CP) (EC 1.11.1.21) (Perox...,Rv1908c,isoniazid,Rv1908c,2153889.0,2156111.0,minus
3,EMB,embB,embCAB.fasta,EMBB_MYCTU,"WHO 2021 catalog, category 1",P9WNL7,Probable arabinosyltransferase B (EC 2.4.2.-),Rv3795,ethambutol,Rv3795,4246513.0,4249809.0,plus
4,PZA,pncA,pncA.fasta,PNCA_MYCTU,"WHO 2021 catalog, category 1",I6XD65,Nicotinamidase/pyrazinamidase (Nicotinamidase)...,Rv2043c,pyrazinamide,Rv2043c,2288681.0,2289241.0,minus
5,LEV,gyrA,gyrBA.fasta,GYRA_MYCTU,"WHO 2021 catalog, category 1",P9WG47,DNA gyrase subunit A (EC 5.6.2.2) (Type IIA to...,Rv0006,levofloxacin,Rv0006,7301.0,9817.0,plus
6,STM,rpsL,rpsL.fasta,RS12_MYCTU,"WHO 2021 catalog, category 1",P9WH63,30S ribosomal protein S12,Rv0682,streptomycin,Rv0682,781559.0,781933.0,plus
7,STM,gid,gid.fasta,RSMG_MYCTU,"WHO 2021 catalog, category 1",P9WGW9,Ribosomal RNA small subunit methyltransferase ...,Rv3919c,streptomycin,Rv3919c,4407528.0,4408202.0,minus
8,ETH,ethA,ethAR.fasta,ETHA_MYCTU,"WHO 2021 catalog, category 1",P9WNF9,FAD-containing monooxygenase EthA (EC 1.14.13....,Rv3854c,ethionamide,Rv3854c,4326004.0,4327473.0,minus
9,BDQ,atpE,NaN,ATPL_MYCTU,NaN,P9WPS1,NaN,Rv1305,bedaquiline,NaN,NaN,NaN,NaN


In [105]:
# Filter the DataFrame for the genes of interest
filtered_df = protein_details[protein_details['gene_name'].str.contains('|'.join(gene_names), case=False, na=False)]

In [106]:
# Initialize an empty list to store DataFrames
all_gene_data = []
for index, row in filtered_df.iterrows():
    # Extract data from the current row
    fasta_filename = row['filename']
    print(f"Processing file: {fasta_filename}")
    uniprot = row['Uniprot']
    entry = row['Entry']
    drug = row['drug_full']
    drug_code = row['Drug']
    gene_name = row['gene_name']
    print("calculating for gene:", gene_name)
    
    ## calculate proximity
    gene_subset=catalog_df[catalog_df['gene'] == gene_name]
    
    ## get r-conferring mutation positions
    r_conferring_mutations = gene_subset[gene_subset['phenotype'] == 'Resistant']['one_letter_mutation'].tolist()
    r_conferring_positions = [extract_position_from_mutation(mutation) for mutation in r_conferring_mutations]
    
    ##load distance file
    distmap_path = f"/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/distmaps/{uniprot}/{entry}"
    dist_map = DistanceMap.from_file(distmap_path)
    
    # Filter valid positions
    valid_r_positions = [pos for pos in r_conferring_positions if pos < dist_map.dist_matrix.shape[0]]
    unique_valid_r_positions=np.unique(valid_r_positions)
    print("len unique Valid R-Conferring Positions:", len(unique_valid_r_positions))
    
    gene_subset=gene_subset.drop_duplicates()

    # # Debug print to check the filtered positions
    print("len Valid R-Conferring Positions:", len(valid_r_positions))
    
    gene_subset=non_self_proximity_r_mutants(gene_subset,unique_valid_r_positions,dist_map)
    # Append the processed DataFrame to the list
    all_gene_data.append(gene_subset)

Processing file: rpoBC.fasta
calculating for gene: rpoB
len unique Valid R-Conferring Positions: 25
len Valid R-Conferring Positions: 90
Processing file: FabG1-inhA.fasta
calculating for gene: inhA
len unique Valid R-Conferring Positions: 1
len Valid R-Conferring Positions: 1
Processing file: KatG.fasta
calculating for gene: katG
len unique Valid R-Conferring Positions: 2
len Valid R-Conferring Positions: 3
Processing file: embCAB.fasta
calculating for gene: embB
len unique Valid R-Conferring Positions: 7
len Valid R-Conferring Positions: 14
Processing file: pncA.fasta
calculating for gene: pncA
len unique Valid R-Conferring Positions: 82
len Valid R-Conferring Positions: 155
Processing file: gyrBA.fasta
calculating for gene: gyrA
len unique Valid R-Conferring Positions: 4
len Valid R-Conferring Positions: 18
Processing file: rpsL.fasta
calculating for gene: rpsL
len unique Valid R-Conferring Positions: 2
len Valid R-Conferring Positions: 4
Processing file: gid.fasta
calculating for ge

In [107]:
# After the loop, concatenate all the DataFrames in the list into a single large DataFrame
final_df = pd.concat(all_gene_data, ignore_index=True)

In [108]:
final_df

,gene,one_letter_mutation,drug,confidence,delta_z,frequency,phenotype,position,Proximity_to_R_Conferring,Nearest_Mutation_Index
0,rpoB,S450L,RIF,1) Assoc w R,0.024894,0.988805,Resistant,450,1.328860,449.0
1,rpoB,D435V,RIF,1) Assoc w R,0.031565,0.987854,Resistant,435,1.324017,434.0
2,rpoB,H445Y,RIF,1) Assoc w R,0.027322,0.988604,Resistant,445,1.329035,446.0
3,rpoB,H445D,RIF,1) Assoc w R,0.023949,0.989691,Resistant,445,1.329035,446.0
4,rpoB,D435Y,RIF,1) Assoc w R,0.034710,0.786408,Resistant,435,1.324017,434.0
...,...,...,...,...,...,...,...,...,...,...
4685,fgd1,L323F,DLM,3) Uncertain significance,0.030537,0.000000,Unknown,323,NaN,NaN
4686,fgd1,K183M,DLM,3) Uncertain significance,0.065629,0.000000,Unknown,183,NaN,NaN
4687,fgd1,R45C,DLM,3) Uncertain significance,0.079128,0.000000,Unknown,45,NaN,NaN
4688,fgd1,M93T,DLM,3) Uncertain significance,0.056434,0.000000,Unknown,93,NaN,NaN


In [109]:
final_df.to_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021_all_proteins_freq_details_proximity.csv",index=False)

## feature 4: amino acid index dist

In [2]:
### essential methods

# Function to extract amino acid and position from mutation string
def extract_aa_and_position(mutation):
    match = re.match(r'([A-Z])(\d+)([A-Z])', mutation)
    if match:
        return match.groups()
    return None, None, None

# Function to calculate the Euclidean distance between two amino acids
def calculate_euclidean_distance(wt_aa, mutant_aa, amino_acid_indices):
    if wt_aa in amino_acid_indices and mutant_aa in amino_acid_indices:
        wt_indices = np.array(amino_acid_indices[wt_aa])
        mutant_indices = np.array(amino_acid_indices[mutant_aa])
        distance = np.linalg.norm(wt_indices - mutant_indices)
        return distance
    else:
        print(f"Missing indices for {wt_aa} or {mutant_aa}")
        return None


In [3]:
catalog_df=pd.read_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021/expanded/2021_all_proteins_freq_details_proximity_aaindex_thermostability.csv")


In [5]:
catalog_df.shape

(4677, 20)

In [6]:
# Load the CSV file into a DataFrame
amino_acid_data_path = '/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/catalog/AAIndex_PCA.csv'  # Update with the correct path
amino_acid_df = pd.read_csv(amino_acid_data_path, index_col=0)

In [ ]:
all_gene_data = []
for index, row in catalog_df.iterrows():
    # Extract data from the current row
    gene_name = row['gene']
    print("calculating for gene:", gene_name)
    
    ## calculate aa index dits
    gene_subset=catalog_df[catalog_df['gene'] == gene_name]
    
    mutations = gene_subset['one_letter_mutation']

    # Create a dictionary to map amino acids to their corresponding index values
    amino_acid_indices = amino_acid_df.set_index(amino_acid_df.index).T.to_dict('list')
    # Calculate distances for each mutation
    distances = []
    for mutation in mutations:
        wt_aa, position, mutant_aa = extract_aa_and_position(mutation)
        if wt_aa and mutant_aa:
            distance = calculate_euclidean_distance(wt_aa, mutant_aa, amino_acid_indices)
            distances.append((distance))
    # Add the distances as a new column to the phenotype data
    gene_subset['aa_index_dist'] = distances
    all_gene_data.append(gene_subset)

In [ ]:
# After the loop, concatenate all the DataFrames in the list into a single large DataFrame
final_df = pd.concat(all_gene_data, ignore_index=True)

### top N components of AA index

In [7]:
# Build dictionary from amino acid -> 19D vector
amino_acid_indices = amino_acid_df.set_index(amino_acid_df.index).T.to_dict('list')

In [ ]:
all_gene_data = []
N_col = 8

for index, row in catalog_df.iterrows():
    gene_name = row['gene']
    print("calculating for gene:", gene_name)
    
    # Subset to just this gene, copy to avoid SettingWithCopy warnings
    gene_subset = catalog_df[catalog_df["gene"] == gene_name].copy()
    mutations = gene_subset["one_letter_mutation"]

    # Prepare lists for the mutant columns
    mutant_cols = [[] for _ in range(N_col)]
    # Prepare lists for the difference columns
    diff_cols = [[] for _ in range(N_col)]

    for mutation in mutations:
        wt_aa, position, mut_aa = extract_aa_and_position(mutation)

        # Lookup vectors (slice to first N columns)
        if wt_aa in amino_acid_indices:
            wt_vec = amino_acid_indices[wt_aa][:N_col]
        else:
            wt_vec = [None]*N_col

        if mut_aa in amino_acid_indices:
            mut_vec = amino_acid_indices[mut_aa][:N_col]
        else:
            mut_vec = [None]*N_col

        # Fill in mutant columns
        for i in range(N_col):
            mutant_cols[i].append(mut_vec[i])

        # Compute the difference: mutant - wildtype
        diff_vec = []
        for i in range(N_col):
            if (wt_vec[i] is not None) and (mut_vec[i] is not None):
                diff_vec.append(mut_vec[i] - wt_vec[i])
            else:
                diff_vec.append(None)

        # Fill in difference columns
        for i in range(N_col):
            diff_cols[i].append(diff_vec[i])

    # Add the new mutant columns
    for i in range(N_col):
        col_name = f"mut_AAIndex{i+1}"
        gene_subset[col_name] = mutant_cols[i]

    # Add the new difference columns
    for i in range(N_col):
        col_name = f"delta_AAIndex{i+1}"
        gene_subset[col_name] = diff_cols[i]

    # Append once
    all_gene_data.append(gene_subset)



In [9]:
# Combine everything at the end
final_df = pd.concat(all_gene_data, ignore_index=True)

In [10]:
final_df.shape

(2335879, 36)

In [11]:
final_df=final_df.drop_duplicates()
final_df.shape

(4677, 36)

In [13]:
final_df.to_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021/expanded/2021_all_proteins_freq_details_proximity_aaindex_2types_thermostability.csv",index=False)

## feature 5: log likelihood ratio

In [29]:
from transformers import AutoTokenizer, EsmForMaskedLM
import torch

In [30]:
catalog_df=pd.read_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021_all_proteins_freq_details_proximity_aaindex.csv")

In [31]:
catalog_df=catalog_df.drop_duplicates()

### debugging fgd1 and pepq

In [ ]:
# Filter rows where proximity is 0.0
proximity_zero_rows = catalog_df[catalog_df['Proximity_to_R_Conferring'] == 'NaN']

# Display the rows with proximity = 0.0
print("Rows where proximity is 0.0:")
print(proximity_zero_rows)

# Get the value counts of phenotype for proximity = 0.0
phenotype_counts = proximity_zero_rows['phenotype'].value_counts()

# Display the counts
print("\nPhenotype value counts where proximity is 0.0:")
print(phenotype_counts)

In [ ]:
# Filter rows where 'Proximity_to_R_Conferring' is NaN
nan_proximity_rows = catalog_df[catalog_df['Proximity_to_R_Conferring'].isna()]

# Print the filtered rows
print(nan_proximity_rows)


In [ ]:
# Count the number of rows with NaN 'Proximity_to_R_Conferring' for each gene
nan_proximity_count = catalog_df[catalog_df['Proximity_to_R_Conferring'].isna()].groupby('gene').size().reset_index(name='NaN Count')

# Print the result
print(nan_proximity_count)


In [ ]:
## filter by gene
## calculate by gene
gene_name = 'pepQ'
genes_of_interest =gene_name.split(',')
print(f"Genes of interest: {genes_of_interest}")
gene_subset=catalog_df[catalog_df['gene'] == gene_name]
## get r-conferring mutation positions
r_conferring_mutations = gene_subset[gene_subset['phenotype'] == 'Resistant']['one_letter_mutation'].tolist()
r_conferring_positions = [extract_position_from_mutation(mutation) for mutation in r_conferring_mutations]
print(len(np.unique(r_conferring_mutations)),len(np.unique(r_conferring_positions)))
print(np.unique(r_conferring_positions))

# # Load the 3D distance map
# if gene_name == 'fgd1':
#     distmap_path = f"/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/proteins_for_mahbuba/FGD1_MYCTU/P9WNE1"
# distance_map = DistanceMap.from_file(distmap_path)


In [ ]:
distance_map.residues_i

In [ ]:
distance_map.residues_i[317:334]

In [ ]:
gene_subset['phenotype'].value_counts()

In [ ]:
np.unique(gene_subset['position'])

In [48]:
gene_subset_sorted = gene_subset.sort_values(by='position', ascending=True)


In [ ]:
# Filter and print the row where 'position' equals target
gene_subset_sorted[90:107]

### calculating llr

In [32]:
catalog_df['Wildtype_AA'] = catalog_df['one_letter_mutation'].str.extract(r'([a-zA-Z])(?=\d)')
catalog_df['Mutated_AA'] = catalog_df['one_letter_mutation'].str.extract(r'(?<=\d)([a-zA-Z])')

In [33]:
protein_sequences_df=pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/catalog/protein_sequences.csv')

In [34]:
# Load the ESM-2 model and tokenizer
model_name = "facebook/esm2_t30_150M_UR50D" ## for rpoB
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)

# List of amino acids
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
start_pos = 1
end_pos = None

In [35]:
# Move the model to GPUs and use DataParallel for multi-GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.nn.DataParallel(model)
model = model.to(device)

In [36]:
catalog_df['gene'].unique()

array(['rpoB', 'inhA', 'katG', 'embB', 'pncA', 'gyrA', 'rpsL', 'gid',
       'ethA', 'atpE', 'ddn', 'Rv0678', 'pepQ', 'rplC', 'tlyA', 'gyrB',
       'fgd1'], dtype=object)

In [37]:
# # Select the rows where 'llr_score' is NaN
# nan_llr_rows = catalog_df['llr_score'].isna()

# # Extract the 'gene' names from those rows
# genes_with_nan_llr = catalog_df.loc[nan_llr_rows, 'gene']

# # Print the gene names
# print("Genes with NaN in 'llr_score':")
# print(genes_with_nan_llr)

# unique_genes_with_nan_llr = genes_with_nan_llr.unique()
# print("Unique genes with NaN in 'llr_score':")
# print(unique_genes_with_nan_llr)


In [ ]:
# Group the catalog by gene to process mutations per gene
for gene_name, gene_subset in catalog_df.groupby('gene'):
    print("Calculating for gene:", gene_name)
    
    # Sort the subset by position
    gene_subset = gene_subset.sort_values(by='position', ascending=True)
    
    # Get the corresponding protein sequence for this gene
    protein_row = protein_sequences_df[protein_sequences_df['gene'] == gene_name]
    
    if not protein_row.empty:
        protein_sequence = protein_row['protein_sequence'].values[0]
        # Tokenize the protein sequence once
        input_ids = tokenizer.encode(protein_sequence, return_tensors="pt").to(device)
        sequence_length = input_ids.shape[1] - 2  # Exclude special tokens

    # Prepare a storage for LLR scores
    llr_scores = []

    # Process each mutation in the gene subset
    for _, row in gene_subset.iterrows():
        position = row['position']  # Ensure 0-based position
        wt_residue = row['Wildtype_AA']
        mt_residue = row['Mutated_AA']
        
        # Mask the target position
        masked_input_ids = input_ids.clone()
        masked_input_ids[0, position + 1] = tokenizer.mask_token_id  # +1 for special token offset

        # Get logits for the masked token
        with torch.no_grad():
            logits = model(masked_input_ids).logits

        # Calculate log probabilities for all residues
        probabilities = torch.nn.functional.softmax(logits[0, position + 1], dim=0)
        log_probabilities = torch.log(probabilities)

        # Convert residues to token IDs
        wt_token_id = tokenizer.convert_tokens_to_ids(wt_residue)
        mt_token_id = tokenizer.convert_tokens_to_ids(mt_residue)

        # Get log probabilities for wild-type and mutant residues
        log_prob_wt = log_probabilities[wt_token_id].item()
        log_prob_mt = log_probabilities[mt_token_id].item()

        # Calculate LLR
        llr_score = log_prob_mt - log_prob_wt
        llr_scores.append((row.name, llr_score))  # Store index and score for later

        print(f"LLR Score for {gene_name} at position {position}: {llr_score}")

    # Update the DataFrame in one go for the current gene
    for index, score in llr_scores:
        catalog_df.at[index, 'llr_score'] = score


In [40]:
# Display the DataFrame with LLR scores
print(catalog_df[['gene', 'one_letter_mutation', 'llr_score']])

         gene one_letter_mutation  llr_score
0        rpoB               S450L   0.525236
1        rpoB               D435V   0.559119
2        rpoB               H445Y   0.440828
3        rpoB               H445D   0.789028
4        rpoB               D435Y  -0.511652
...       ...                 ...        ...
2343137  fgd1               L323F   0.314339
2343138  fgd1               K183M   5.846107
2343139  fgd1                R45C  -4.410552
2343140  fgd1                M93T   2.363091
2343141  fgd1                R64S   5.875133

[4690 rows x 3 columns]


In [41]:
catalog_df.to_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/2021_all_proteins_freq_details_proximity_aaindex_llr.csv",index=False)

## feature 6: thermostability score

In [2]:
catalog_df=pd.read_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021/2021_all_proteins_freq_details_proximity_aaindex_llr.csv")

In [3]:
protein_sequences_df=pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/catalog/protein_sequences.csv')

In [ ]:
all_gene_data = []
# Group the catalog by gene to process mutations per gene
for gene_name, gene_subset in catalog_df.groupby('gene'):
    print("Calculating for gene:", gene_name)
    
    # Sort the subset by position
    gene_subset = gene_subset.sort_values(by='position', ascending=True)
    # Load the Rosetta score CSV
    if gene_name=='gyrA':
        # Define file paths for N-terminal and C-terminal
        nter_file_path = f'/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/Rosetta/refined/{gene_name}_Nter_ddG.csv'
        cter_file_path = f'/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/Rosetta/refined/{gene_name}_Cter_ddG.csv'
        # Load the two parts
        nter_df = pd.read_csv(nter_file_path)
        cter_df = pd.read_csv(cter_file_path)
        # Concatenate the two DataFrames
        rosetta_scores= pd.concat([nter_df, cter_df], ignore_index=True)
    else:
        rosetta_scores_path = f'/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/Rosetta/refined/{gene_name}_ddG.csv' 
        rosetta_scores = pd.read_csv(rosetta_scores_path)
    rosetta_scores['Wildtype_AA'] = rosetta_scores['variant'].str.extract(r'([a-zA-Z])(?=\d)')
    rosetta_scores['position'] = rosetta_scores['variant'].str.extract(r'(\d+)').astype(int)
    rosetta_scores['Mutated_AA'] = rosetta_scores['variant'].str.extract(r'(?<=\d)([a-zA-Z])')
    # Merge the dataframes on Adjusted_Number and One_Letter
    common_data = pd.merge(
        rosetta_scores,
        gene_subset,
        left_on=['Wildtype_AA', 'position','Mutated_AA'],
        right_on=['Wildtype_AA', 'position','Mutated_AA'],
        how='inner'
        )
    print(common_data.columns)
    # common_data=common_data.drop(columns=['fa_intra_rep', 'fa_intra_sol_xover4',
    #    'lk_ball_wtd', 'pro_close', 'hbond_sr_bb', 'hbond_lr_bb',
    #    'hbond_bb_sc', 'hbond_sc', 'dslf_fa13', 'omega', 'p_aa_pp',
    #    'yhh_planarity', 'ref', 'rama_prepro','variant'])
    common_data=common_data.drop(columns=['fa_intra_rep', 'fa_intra_sol_xover4',
       'lk_ball_wtd', 'pro_close', 'hbond_sr_bb', 'hbond_lr_bb',
       'hbond_bb_sc', 'hbond_sc', 'omega', 'p_aa_pp', 'ref', 'rama_prepro','variant'])
    common_data=common_data.rename(columns={"score": "thermostability"})
    cols = ['gene'] + [col for col in common_data.columns if col != 'gene']
    common_data = common_data[cols]
    common_data.to_csv(f'/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021/expanded_rosetta/{gene_name}_thermostability.csv', index=False)
    all_gene_data.append(common_data)

In [10]:
# After the loop, concatenate all the DataFrames in the list into a single large DataFrame
final_df = pd.concat(all_gene_data, ignore_index=True)
final_df.to_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/2021/expanded/2021_all_proteins_freq_details_proximity_aaindex_thermostability.csv",index=False)

In [284]:
# # Create a function to adjust the number based on conditions
# def adjust_number(index):
#     if index < 30:  # Assuming 23 + 7
#         return index- 7
#     elif 30 <= index <= 1180:  # Assuming 1173 + 7
#         return index - 6
#     else:
#         return index

# #code for pdb data with position adjustment
# gene_name = 'pncA'
# # Load the Rosetta score CSV
# rosetta_scores_path = f'/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/Rosetta/{gene_name}_ddG.csv'  # Update with the correct path
# rosetta_scores = pd.read_csv(rosetta_scores_path)
# # Select the subset of rows for the current gene
# gene_subset = catalog_df[catalog_df['gene'] == gene_name]
# gene_subset= gene_subset.sort_values(by='position', ascending=True)
# gene_subset
# rosetta_scores['Wildtype_AA'] = rosetta_scores['variant'].str.extract(r'([a-zA-Z])(?=\d)')
# rosetta_scores['position'] = rosetta_scores['variant'].str.extract(r'(\d+)').astype(int)
# rosetta_scores['Mutated_AA'] = rosetta_scores['variant'].str.extract(r'(?<=\d)([a-zA-Z])')
# # Apply the adjustment to the 'Number' in pdb_df
# rosetta_scores['adjusted_position'] = rosetta_scores['position'].apply(adjust_number)
# # Merge the dataframes on Adjusted_Number and One_Letter
# common_data = pd.merge(
#     rosetta_scores,
#     gene_subset,
#     left_on=['Wildtype_AA', 'adjusted_position','Mutated_AA'],
#     right_on=['Wildtype_AA', 'position','Mutated_AA'],
#     how='inner'
# )
# common_data=common_data.drop(columns=['fa_atr', 'fa_rep', 'fa_sol', 'fa_intra_rep', 'fa_intra_sol_xover4',
#        'lk_ball_wtd', 'fa_elec', 'pro_close', 'hbond_sr_bb', 'hbond_lr_bb',
#        'hbond_bb_sc', 'hbond_sc', 'dslf_fa13', 'omega', 'fa_dun', 'p_aa_pp',
#        'yhh_planarity', 'ref', 'rama_prepro','variant','position_x'])
# common_data=common_data.rename(columns={"score": "thermostability"})
# # Move 'gene' column to the front
# cols = ['gene'] + [col for col in common_data.columns if col != 'gene']
# common_data = common_data[cols]
# common_data
# common_data.to_csv(f'/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/{gene_name}_thermostability.csv', index=False)